In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   LogisticRegression
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"E:\GCaMP6s_EX357")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── GCaMP6s_EX357
    ├── GCaMP6s_EX357_4Conditions
    │   ├── Week 1
    │   │   ├── 1-1-1
    │   │   ├── 1-1-2_0001
    │   │   ├── 1-1-3
    │   │   ├── 1-2-1
    │   │   ├── 1-2-2
    │   │   ├── 1-2-3
    │   │   ├── 1-3-1
    │   │   ├── 1-3-2
    │   │   ├── 1-3-3
    │   │   ├── 1-4-1
    │   │   ├── 1-4-2
    │   │   ├── 1-4-3
    │   │   └── metrics
    │   ├── Week 2
    │   │   ├── 2-1-1
    │   │   ├── 2-1-2
    │   │   ├── 2-1-3
    │   │   ├── 2-2-1
    │   │   ├── 2-2-2
    │   │   ├── 2-2-3
    │   │   ├── 2-3-1
    │   │   ├── 2-3-2
    │   │   ├── 2-3-3
    │   │   ├── 2-4-1
    │   │   ├── 2-4-2
    │   │   ├── 2-4-3
    │   │   └── metrics
    │   └── metrics
    ├── GCaMP6s_EX357_DL-AP5
    │   ├── Week 2 25uM
    │   │   ├── 2-1
    │   │   ├── 2-1_25uM_8m
    │   │   ├── 2-2
    │   │   ├── 2-2_25uM_2m
    │   │   ├── 2-3
    │   │   ├── 2-3_25uM_14m
    │   │   └── metrics
    │   ├── Week 2 50uM
    │   │   ├── 2-1
    │   │   ├── 2-1_50uM_2m
    │   │   ├──

In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 1-1-1
  Traces: 372 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 282/372 kept (75.8%)
  Spikes: 3031/27074 kept | neurons 282 → 281
  Grouping (corr+sttc): | corr=17 | sttc=1 | corr_vs_sttc=0.85

 Processing: 1-1-2_0001
  Traces: 469 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 294/469 kept (62.7%)
  Spikes: 2817/29169 kept | neurons 294 → 294
  Grouping (corr+sttc): | corr=27 | sttc=0 | corr_vs_sttc=0.78

 Processing: 1-1-3
  Traces: 480 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 385/480 kept (80.2%)
  Spikes: 5043/36491 kept | neurons 385 → 385
  Grouping (corr+sttc): | corr=24 | sttc=1 | corr_vs_sttc=0.86

 Processing: 1-2-1
  Traces: 378 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 324/378 kept (85.7%)
  Spikes: 5026/30384 kept | neurons 324 → 324
  Grouping (corr+sttc): | corr=33 | sttc=1 | corr_vs_sttc=0.76

 Processing: 1-2-2
  Traces: 603 ROIs, 3650 frames @ 15.0 Hz
  ROI filter: 475/603 kept (78.8%)
  Spikes: 6907/44848 kept | neurons 475 → 474
  Grouping (corr+sttc): | co

In [6]:
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: E:\GCaMP6s_EX357
                    child  n_videos  n_neurons  n_groups_corr  n_groups_sttc  frac_grouped  frac_ungrouped  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_grouped  half_max_width_seconds_mean_grouped  rise_slope_hz_mean_grouped  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_grouped  spike_frequency_mean_ungrouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds_between_unweighted  rise_slope_hz_var_unweighted  rise_slope_hz_within_unweighted  rise_slope_hz_between_unweighted  decay_

In [8]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)